# Silver Table Inspection

Ad-hoc scan of `raw_data/silver/picks`, the Delta table written by the `silver_picks` Dagster asset (`sts_pipeline/assets/silver.py`) — one row per card-reward pick-event, filtered to standard runs and enriched with confounders. Dev/inspection notebook, not part of the numbered analysis sequence.

In [1]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

SILVER_PICKS_PATH = str(PROJECT_ROOT / "raw_data" / "silver" / "picks")

In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder.master("local[*]")
    .appName("silver-inspection")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "8g")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(SILVER_PICKS_PATH)
print(f"Loaded {SILVER_PICKS_PATH}")

Loaded e:\Projects\sts-card-choice-analysis\raw_data\silver\picks


## Schema

One row per pick-event (not per run). `not_picked_options` is the only remaining nested (array) column — everything else was flattened during the silver transform.

In [3]:
df.printSchema()

root
 |-- play_id: string (nullable = true)
 |-- pick_num: integer (nullable = true)
 |-- card_selected: string (nullable = true)
 |-- card_selected_floor: integer (nullable = true)
 |-- card_not_selected: boolean (nullable = true)
 |-- card_not_selected_floor: integer (nullable = true)
 |-- not_picked_options: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- floor_reached: long (nullable = true)
 |-- victory: boolean (nullable = true)
 |-- character_chosen: string (nullable = true)
 |-- ascension_level: long (nullable = true)
 |-- current_hp: long (nullable = true)
 |-- max_hp: long (nullable = true)
 |-- relic_count: integer (nullable = true)
 |-- neow_bonus: string (nullable = true)
 |-- neow_cost: string (nullable = true)
 |-- player_experience: long (nullable = true)
 |-- floors_gained: long (nullable = true)



## Row count and sanity checks

Checks the `card` / `is_selected` invariant (card populated iff is_selected=True) — the same mutual-exclusivity idea as `02_data_preparation.ipynb`'s "Check for Mutual Exclusivity" section, adapted to the renamed columns.

In [ ]:
from pyspark.sql import functions as F

print("Row count:", df.count())

missing_card = df.filter(F.col("is_selected") & F.col("card").isNull()).count()
unexpected_card = df.filter((~F.col("is_selected")) & F.col("card").isNotNull()).count()
print("Rows marked is_selected=True but card is null:", missing_card)
print("Rows marked is_selected=False (skip) but card is populated:", unexpected_card)

skip_rate = df.agg(F.mean((~F.col("is_selected")).cast("double"))).first()[0]
print("Skip rate:", skip_rate)

df.select("floors_gained").summary().show()

## Table view

`.toPandas()` on a small slice, every column included (including `not_picked_options`) so nothing is hidden from view.

In [ ]:
df.limit(50).toPandas()

## Most-picked cards

Just a shape check, not real analysis (that's gold's job) — confirms `card` values look like real card names at sane frequencies.

In [ ]:
df.filter(F.col("card").isNotNull()) \
    .groupBy("card") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()